## A more visible Agent loop

The digital Twin conatined an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't  feel very loopy.

### Adding more ingredient to make it more real

Let's make an agent loop with some familiar features borrowed from Cluade Code.

1- A Terminal UI
2- A Checklist tool to cause and track multiple calls

In [1]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os

load_dotenv(override=True)





True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
show("Bilal")

Bilal

In [4]:
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [6]:
checklist = []
completed = []

def get_checklist_report()-> str:
    result=""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index+1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index+1}: [red]{item}[/red]\n"
    
    show(result)
    return result




In [9]:
get_checklist_report()

''

In [7]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [16]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [12]:
create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [red]Buy groceries[/red]\nChecklist #2: [red]Finish week 1[/red]\nChecklist #3: [red]Eat banana[/red]\n'

In [13]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: [red]Finish week 1[/red]\nChecklist #3: [red]Eat banana[/red]\n'

In [9]:

create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}



In [10]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [11]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [12]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [13]:
def loop(messages):
    response = ollama.chat.completions.create(model="qwen2.5:7b", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = ollama.chat.completions.create(model="qwen2.5:7b", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [14]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [15]:
loop(messages)

TypeError: create_checklist() got an unexpected keyword argument 'index'